# 03 — Fitting I: what we optimize (log-loss)

*Notebook 3 of 6 — Logistic Regression*

In NB 1 and NB 2 we *chose* the weights by hand and looked at the result. To let a computer **find**
them, we first need a single number that says how *wrong* a choice of weights is — a **loss**. This
notebook builds that number, and shows why one particular loss is the right one for classification.

**Prerequisites.** NB 1: the sigmoid, and that the score $z$ *is* the log-odds. NB 2: the weighted score
$z = w_1 x_1 + w_2 x_2 + b$, set by hand. Chapter 02: the **likelihood** (we maximized it by counting),
and working in **log-space** (NB 03).

**What you'll be able to do.**
- Define the **log-loss** (cross-entropy) and recognise it as the **negative log-likelihood**.
- Explain why it punishes a confident-and-wrong prediction hardest.
- Say why squared error is a poor training objective here (non-convex, with stalling plateaus).
- Read a loss curve and rank weight choices by their loss.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course import colors, datasets, viz

viz.use_course_style()
np.random.seed(0)  # no randomness here, but we keep the habit


def sigmoid(z):
    """Map any real score z to a probability in (0, 1)."""
    return 1.0 / (1.0 + np.exp(-z))


# One feature, standardized by hand (z-score) — the same standardization as NB 2.
df = datasets.load_penguins()
bill = df["bill_length_mm"].to_numpy(dtype=float)
x = (bill - bill.mean()) / bill.std()  # numpy std is population (ddof=0), as StandardScaler uses
y = (df["species"] == "Gentoo").to_numpy().astype(float)
print(f"{len(y)} penguins | Gentoo (y=1): {int(y.sum())}")
print(f"bill mean {bill.mean():.2f} | std {bill.std():.2f}")

## Where we are

In NB 1 and NB 2 we **picked** the weights ourselves and judged them by eye. A computer cannot "judge by
eye"; it needs a number to drive down. So we define a **loss**: a single value, computed from the
weights and the data, that is **small when the weights are good and large when they are bad**. Fitting is
then nothing more than *finding the weights that make the loss smallest*.

This notebook builds that loss and studies its shape. The next one (NB 4) does the minimizing. To keep
the picture to a single curve, we hold the bias at $b = 0$ and vary one weight $w$ — still **nothing
fitted**; we are only *measuring* how wrong each choice is.

## From likelihood to a loss

In chapter 02 we scored a model by its **likelihood** — the probability it assigns to the labels we
actually observed. For one penguin the model gives $P = \sigma(z)$ for Gentoo; if the bird really is
Gentoo ($y = 1$) we want $P$ high, and if it is Adélie ($y = 0$) we want $1 - P$ high. Good weights make
the **true label's** probability large.

The likelihood of the whole dataset is the **product** of those per-penguin probabilities. Two small
moves turn "maximize a product" into "minimize a sum":

1. take the **logarithm** — a product becomes a sum (the log-space trick from chapter 02, NB 03), much
   easier to handle;
2. take the **negative** — so that *smaller is better*, a quantity to push **down**.

The result, per penguin, is the **log-loss** (also called **cross-entropy**):

$$ \ell(y, P) = -\big[\, y \log P + (1 - y)\log(1 - P) \,\big]. $$

It is exactly the **negative log-likelihood** of the **Bernoulli model** — the model that treats each
label as one biased coin-flip (Gentoo with probability $P$, Adélie with $1 - P$). This is chapter 02's
likelihood, now read as a loss to minimize.

In [ ]:
def log_loss_one(y, p):
    """Log-loss (cross-entropy) of one example: -[y log p + (1-y) log(1-p)]."""
    eps = 1e-12  # guard the logarithm away from 0
    return -(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))


pairs = [(1, 0.9), (1, 0.5), (1, 0.1), (0, 0.1), (0, 0.9)]
pd.DataFrame(
    {
        "true y": [yi for yi, _ in pairs],
        "model P(Gentoo)": [pi for _, pi in pairs],
        "log-loss": [log_loss_one(yi, pi) for yi, pi in pairs],
    }
).round(3)

**Read the table.** The log-loss is near **0** when the model is confident **and** right (true Gentoo
at $P = 0.9$, or true Adélie at $P = 0.1$, both cost $0.105$). It is **large** when the model is
confident **and** wrong (true Gentoo at $P = 0.1$ costs $2.303$). A useful way to hear it: the more the
*truth* should **surprise** the model, the higher the loss. The two classes are treated symmetrically.

## Confident and wrong is punished hardest

Take one true Gentoo ($y = 1$); its log-loss is $-\log P$. As the model's $P$ slides toward 0 —
confidently calling a true Gentoo an Adélie — $-\log P$ grows **without bound**: $P = 0.1$ costs $2.3$,
$P = 0.01$ costs $4.6$, $P = 0.001$ costs $6.9$, and it never stops climbing.

Compare **squared error** $(P - y)^2$. For the same true Gentoo it is $(P - 1)^2$, which can never
exceed **1**, no matter how confidently wrong the model is. A loss that caps its own penalty barely
discourages a confident mistake. Log-loss does not cap it — and that is what pushes the model toward
**honest** probabilities.

In [ ]:
grid_p = np.linspace(0.001, 0.999, 500)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(
    grid_p, -np.log(grid_p), color=colors.COLORS["model"], linewidth=2, label="log-loss   −log P"
)
ax.plot(
    grid_p,
    (grid_p - 1) ** 2,
    color=colors.COLORS["highlight"],
    linewidth=2,
    label="squared error   (P − 1)²",
)
ax.scatter([0.05], [-np.log(0.05)], color=colors.COLORS["model"], zorder=5, s=55)
ax.annotate(
    "confident & wrong",
    xy=(0.05, -np.log(0.05)),
    xytext=(0.30, 4.3),
    color=colors.COLORS["text"],
    arrowprops=dict(arrowstyle="->", color=colors.COLORS["text"]),
)
ax.set_xlabel("P(Gentoo) the model gave to a true Gentoo (y = 1)")
ax.set_ylabel("penalty")
ax.set_title("How each loss punishes a true y = 1")
ax.legend()
plt.show()

**Read the figure.** Reading right to left, as a confident prediction turns out wrong ($P \to 0$) the
**log-loss** (blue) shoots up without limit, while the **squared error** (rose) barely passes 1 and then
flattens. The steep left wall is the model being held to account for false confidence — and it is exactly
the behaviour we want a training signal to have.

## The loss over the whole dataset, as a function of the weight

So far, one penguin at a time. The loss of a *set of weights* is the **average** of the per-penguin
log-loss over all 274 birds. With the bias held at $b = 0$ and the single weight $w$ free, that average
is a function $L(w)$ — one number for each choice of $w$. **Fitting** will mean *finding the $w$ at the
bottom of $L$*. Let us plot it, and beside it the curve squared error would give.

In [ ]:
def total_log_loss(w, b=0.0):
    p = sigmoid(w * x + b)
    return np.mean(log_loss_one(y, p))


def total_squared_error(w, b=0.0):
    p = sigmoid(w * x + b)
    return np.mean((p - y) ** 2)


grid_w = np.linspace(-6, 20, 600)
ll = np.array([total_log_loss(w) for w in grid_w])
se = np.array([total_squared_error(w) for w in grid_w])

# Convexity, read off the discrete second difference; and the slope far out on the right.
dw = grid_w[1] - grid_w[0]
print(f"log-loss : min L={ll.min():.3f} at w={grid_w[ll.argmin()]:.2f} | "
      f"2nd-diff min={np.diff(ll, 2).min():+.2e}  (>= 0  => convex)")
print(f"sq-error : min L={se.min():.3f} at w={grid_w[se.argmin()]:.2f} | "
      f"2nd-diff min={np.diff(se, 2).min():+.2e}  (< 0  => non-convex)")
print(f"slope at w=20 : log-loss {(ll[-1] - ll[-2]) / dw:+.2e}   "
      f"sq-error {(se[-1] - se[-2]) / dw:+.2e}  (sq-error ~ flat plateau)")

hand_w = [1.0, 3.0, 6.2]
hand_ll = [total_log_loss(w) for w in hand_w]

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.6))

axL.plot(grid_w, ll, color=colors.COLORS["model"], linewidth=2)
axL.scatter(hand_w, hand_ll, color=colors.COLORS["highlight"], zorder=5, s=55)
for w_i, l_i in zip(hand_w, hand_ll, strict=True):
    axL.annotate(
        f"w={w_i:g}",
        xy=(w_i, l_i),
        xytext=(w_i, l_i + 0.13),
        ha="center",
        color=colors.COLORS["text"],
    )
axL.scatter(
    [grid_w[ll.argmin()]], [ll.min()], color=colors.COLORS["model"], marker="v", s=80, zorder=6
)
axL.set_ylim(0, 2.0)
axL.set_xlabel("weight w   (bias held at b = 0)")
axL.set_ylabel("mean log-loss  L(w)")
axL.set_title("Log-loss: a single convex bowl")

axR.plot(grid_w, se, color=colors.COLORS["highlight"], linewidth=2)
axR.scatter(
    [grid_w[se.argmin()]], [se.min()], color=colors.COLORS["highlight"], marker="v", s=80, zorder=6
)
axR.annotate(
    "flat plateau (slope ≈ 0)",
    xy=(19, total_squared_error(19.0)),
    xytext=(2.0, 0.62 * se.max()),
    color=colors.COLORS["text"],
    arrowprops=dict(arrowstyle="->", color=colors.COLORS["text"]),
)
axR.set_xlabel("weight w   (bias held at b = 0)")
axR.set_ylabel("mean squared error  L(w)")
axR.set_title("Squared error: non-convex, with plateaus")

plt.tight_layout()
plt.show()

**Read the figure.** *Left:* log-loss is a single smooth **bowl**. There is one bottom (the marked
$\triangledown$), and the walls keep rising — steeply on the wrong side, climbing past the top of this
view — so from any starting weight, walking *downhill* leads to the one best value. The three rose dots are three weight choices; the lowest
one sits nearest the bottom. *Right:* squared error is **non-convex** — it dips to a minimum but then
**flattens into plateaus** at both ends (where $\sigma$ saturates and the probabilities stop moving), so
its slope nearly vanishes far from the optimum and a downhill search there **stalls**. (Honest detail: on
this 1-D data each curve has a single lowest point; what differs is the **shape** — a convex bowl with a
live slope everywhere, versus a non-convex curve with dead-flat plateaus.)

## Why convexity is the prize

A **convex** loss has one simple shape: a single bottom, no flat traps, no separate valleys. Start
anywhere, keep stepping downhill, and you arrive at the one best set of weights. That guarantee is what
makes the next notebook's **gradient descent** dependable.

**Log-loss is convex** for logistic regression — a known result (ESL §4.4; Bishop §4.3.2), and one we can
*see* in the bowl above: its curvature never turns the wrong way. **Squared error through the sigmoid is
not convex**, which is why classification is trained with log-loss rather than the squared error we might
borrow from regression.

In [ ]:
choices = [1.0, 3.0, 6.2]
ranked = pd.DataFrame(
    {
        "weight w (bias 0)": choices,
        "mean log-loss": [total_log_loss(w) for w in choices],
    }
).round(4)
ranked.sort_values("mean log-loss").reset_index(drop=True)

**Read the result.** Lower loss means better weights — that is the whole point. In NB 2 we set weights
by hand with no way to say which choice was better; now a single number ranks them, and the best of these
three ($w = 6.2$) sits at the bottom of the bowl. NB 4 will *find* that bottom for us, for $w$ **and** $b$
together, instead of trying values by hand.

## Your turn

1. **Rank them.** Three weight settings score log-loss $0.41$, $0.20$, and $0.146$. Which weights are
   best, and which will the fit prefer? (One word: lower or higher loss?)
2. **Why the explosion?** Using $-\log P$, explain in a sentence why a model that says $P = 0.01$ for a
   true Gentoo is punished far more than one that says $P = 0.45$. What does squared error do instead?
3. **By hand.** A penguin gets $P = 0.8$. Compute its log-loss **if it is truly Gentoo** ($y = 1$) and
   **if it is truly Adélie** ($y = 0$), using $\ell = -[y\log P + (1-y)\log(1-P)]$. Check both against
   `log_loss_one`.

## What you built

- The **log-loss** $\ell(y,P) = -[y\log P + (1-y)\log(1-P)]$ — the **cross-entropy**, equal to the
  **negative log-likelihood** of the Bernoulli model (chapter 02's likelihood, read as a loss).
- Why it **punishes confident-and-wrong without bound**, where squared error caps out at 1.
- Why it is the right objective: it is **convex** (one bottom, reachable downhill from anywhere), unlike
  squared error through the sigmoid, which is **non-convex with stalling plateaus**.
- A single number that **ranks** any two weight choices.

**Vocabulary.** *loss / objective* · *likelihood* · *negative log-likelihood* · *cross-entropy /
log-loss* · *convex* · *plateau*.

You now have the target. Next we go and **reach** it — rolling downhill to the bottom of the bowl.

## Going further (optional)

There is a small miracle waiting in NB 4: the gradient of the log-loss with respect to the score is
exactly $P - y$, and with respect to a weight it is $(P - y)\,x$ — the sigmoid's derivative **cancels**.
That clean form is what makes gradient descent on log-loss stable and quick. Squared error's gradient
keeps an extra $\sigma'(z)$ factor that goes to zero exactly on the saturating plateaus we saw — so a
descent there receives almost no signal and crawls.

A look ahead: **NB 4** walks this convex bowl downhill with **gradient descent**, and watches the weights
roll to the bottom.

## References

- Cox, D. R. (1958). *The regression analysis of binary sequences.* Journal of the Royal Statistical
  Society B, 20(2), 215–242. DOI: 10.1111/j.2517-6161.1958.tb00292.x
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*, §4.3.2 (logistic regression, the
  cross-entropy error and its gradient). Springer.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (§4.4,
  logistic regression and its log-likelihood). DOI: 10.1007/978-0-387-84858-7

---

*Previous: 02 — The decision boundary & reading the weights*  ·  *Next: 04 — Fitting II: gradient
descent*